<a href="https://colab.research.google.com/github/Asaf21S/constrained-flow-matching/blob/main/constrained-flow-matching/constraints_distillation/notebooks/constrained_fm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
!pip install torchcfm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
from torchcfm.conditional_flow_matching import ConditionalFlowMatcher, ExactOptimalTransportConditionalFlowMatcher
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import math
from scipy.stats import wasserstein_distance, ks_2samp

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# Dataset

In [ ]:
RNG = np.random.default_rng()

In [ ]:
def get_1d_gaussian_data(peaks: list[tuple[float, float]], n_samples: int, rng=RNG):
    means = np.array([p[0] for p in peaks])
    stds = np.array([p[1] for p in peaks])
    weights = np.full(len(peaks), 1.0 / len(peaks))

    # Pick which peak each sample belongs to
    choices = rng.choice(len(peaks), p=weights, size=n_samples)

    # Generate standard normal samples (mean=0, std=1)
    base_samples = rng.standard_normal(size=n_samples)

    # Scale and shift the base samples
    mixture_samples = (base_samples * stds[choices]) + means[choices]

    return torch.from_numpy(mixture_samples).float()

In [ ]:
peaks = [(2, 1), (-2, 2)]
peaks_str = "".join([f"_{mean}_{std}" for mean, std in peaks])

In [ ]:
real_data = get_1d_gaussian_data(peaks=peaks, n_samples=100000)

In [ ]:
# plot data:
plt.hist(real_data, bins=100)
plt.show()

# General Functions

## visualization

In [ ]:
def _plot_trajectories(ax, sampled_data, boundaries=None):
    ax.set_title("1D Trajectories Over Time")
    start_points, end_points = sampled_data[0], sampled_data[-1]
    n_steps, n_samples = sampled_data.shape
    n_plot = min(100, n_samples)
    t_values = np.linspace(0, 1, n_steps)

    ax.plot(t_values, sampled_data[:, :n_plot], color='black', alpha=0.15)
    ax.scatter(np.zeros(n_plot), start_points[:n_plot], c='red', s=15, label='Start (Noise)', zorder=3)
    ax.scatter(np.ones(n_plot), end_points[:n_plot], c='blue', s=15, label='End (Data)', zorder=3)

    if boundaries is not None:
        min_val, max_val = boundaries
        ax.axhline(y=min_val, color='green', linestyle='--', linewidth=2, zorder=4, label='Boundaries')
        ax.axhline(y=max_val, color='green', linestyle='--', linewidth=2, zorder=4)

    ax.set_xlabel("Time (t)")
    ax.set_ylabel("Position (x)")
    ax.legend()
    ax.grid(True, alpha=0.3)

In [ ]:
def _plot_macro(ax, end_points, title, true_data=None, boundaries=None, secondary_ends=None, primary_label="With Hardflow", secondary_label="Without Hardflow", alpha=0.5):
    n_samples = len(end_points)
    macro_title = title

    if boundaries is not None:
        min_val, max_val = boundaries

        # Primary Metrics
        inside_count = ((end_points >= min_val) & (end_points <= max_val)).sum()
        macro_title += f"\n{primary_label}: {inside_count / n_samples * 100:.2f}%"

        # Secondary Metrics
        if secondary_ends is not None:
            n_sec_samples = len(secondary_ends)
            sec_inside_count = ((secondary_ends >= min_val) & (secondary_ends <= max_val)).sum()
            macro_title += f"\n{secondary_label}: {sec_inside_count / n_sec_samples * 100:.2f}%"

        ax.axvline(x=min_val, color='green', linestyle='--', linewidth=2, zorder=4)
        ax.axvline(x=max_val, color='green', linestyle='--', linewidth=2, zorder=4)

    ax.set_title(macro_title)

    if true_data is not None:
        ax.hist(true_data, bins=100, density=True, color='magenta', alpha=0.2, label="Original Prior")

    # Primary Filled Histogram
    ax.hist(end_points, bins=100, density=True, color='blue', alpha=alpha, label=primary_label)

    # Secondary Step Histogram (Outline only to avoid visual clutter)
    # if secondary_ends is not None:
    #     ax.hist(secondary_ends, bins=100, density=True, histtype='step',
    #             color='darkorange', linewidth=2, linestyle='--', zorder=6, label=secondary_label)

    ax.set_xlabel("Position (x)")
    ax.set_ylabel("Density")
    ax.legend()
    ax.grid(True, alpha=0.3)

In [ ]:
def _plot_micro(ax, end_points, true_data=None, boundaries=None, secondary_ends=None, primary_label="With Hardflow", secondary_label="Without Hardflow", alpha=0.5):
    min_val, max_val = boundaries
    micro_title = "Zoom: Conditional Fit (Shape Matching)"

    # Filter to only show points that made it inside the bounds
    gen_inside = end_points[(end_points >= min_val) & (end_points <= max_val)]
    ax.hist(gen_inside, bins=100, density=True, color='blue', alpha=alpha, label=primary_label)

    if secondary_ends is not None:
        sec_inside = secondary_ends[(secondary_ends >= min_val) & (secondary_ends <= max_val)]
        # Use a distinct dashed step-plot for the secondary model to avoid visual mud
        ax.hist(sec_inside, bins=100, density=True, histtype='step',
                color='darkorange', linewidth=2, linestyle='--', zorder=6, label=secondary_label)

    if true_data is not None:
        ideal_dist = true_data[(true_data >= min_val) & (true_data <= max_val)]
        ax.hist(ideal_dist, bins=100, density=True, histtype='step',
                color='black', linewidth=2, linestyle='-', zorder=5, label="Ideal Target")

        # Calculate Primary Metrics
        w_dist, ks_stat = calculate_distribution_metrics(end_points, ideal_dist)
        micro_title += f"\n{primary_label}   -> W-Dist: {w_dist:.4f} | KS: {ks_stat:.4f}"

        if secondary_ends is not None:
            w_dist2, ks_stat2 = calculate_distribution_metrics(secondary_ends, ideal_dist)
            micro_title += f"\n{secondary_label} -> W-Dist: {w_dist2:.4f} | KS: {ks_stat2:.4f}"

    ax.set_title(micro_title)
    ax.axvline(x=min_val, color='green', linestyle='--', linewidth=2, zorder=4)
    ax.axvline(x=max_val, color='green', linestyle='--', linewidth=2, zorder=4)

    margin = (max_val - min_val) * 0.1
    ax.set_xlim(min_val - margin, max_val + margin)
    ax.set_xlabel("Position (x)")
    ax.legend()
    ax.grid(True, alpha=0.3)

In [ ]:
def visualize_fm(sampled_data, title="1D Flow Matching Results", alpha=0.5, true_data=None,
                 boundaries=None, secondary_sampled_data=None, primary_label="With Hardflow", secondary_label="Without Hardflow"):
    """
    Visualizes Flow Matching trajectories and final distributions for 1D data.
    Dynamically creates a 3rd subplot for zoomed-in boundary analysis if boundaries are provided.
    Accepts an optional secondary dataset to compare on the zoomed-in micro plot.
    """
    # Format Primary Data
    if sampled_data.ndim == 3 and sampled_data.shape[-1] == 1:
        sampled_data = sampled_data.squeeze(-1)
    end_points = sampled_data[-1]

    # Format Secondary Data (if provided)
    secondary_ends = None
    if secondary_sampled_data is not None:
        if secondary_sampled_data.ndim == 3 and secondary_sampled_data.shape[-1] == 1:
            secondary_sampled_data = secondary_sampled_data.squeeze(-1)
        secondary_ends = secondary_sampled_data[-1]

    # Format True Data
    if true_data is not None and true_data.ndim == 2 and true_data.shape[-1] == 1:
        true_data = true_data.squeeze(-1)

    num_plots = 3 if boundaries is not None else 2
    fig, axes = plt.subplots(1, num_plots, figsize=(6 * num_plots, 5))

    if num_plots == 2:
        ax_traj, ax_macro = axes
        ax_micro = None
    else:
        ax_traj, ax_macro, ax_micro = axes

    # Dispatch to helpers
    _plot_trajectories(ax_traj, sampled_data, boundaries)
    _plot_macro(ax_macro, end_points, title, true_data, boundaries, secondary_ends, primary_label, secondary_label, alpha)

    if ax_micro is not None:
        _plot_micro(ax_micro, end_points, true_data, boundaries, secondary_ends, primary_label, secondary_label, alpha)

    plt.tight_layout()
    plt.show()

## Metrics

In [ ]:
def calculate_distribution_metrics(generated_data, true_data):
    """
    Calculates the Wasserstein Distance and KS Statistic between two 1D distributions.

    Args:
        generated_data: The 1D array/tensor of generated points (e.g., at t=1).
        true_data: The 1D array/tensor of ideal target points.

    Returns:
        w_dist (float): Wasserstein distance (closer to 0 is better).
        ks_stat (float): KS Statistic, bounded [0, 1] (closer to 0 is better).
    """
    if hasattr(generated_data, 'cpu'):
        generated_data = generated_data.detach().cpu().numpy()
    if hasattr(true_data, 'cpu'):
        true_data = true_data.detach().cpu().numpy()

    gen_flat = generated_data.flatten()
    true_flat = true_data.flatten()

    # Wasserstein Distance
    w_dist = wasserstein_distance(gen_flat, true_flat)

    # Kolmogorov-Smirnov Statistic
    ks_stat, _ = ks_2samp(gen_flat, true_flat)

    return w_dist, ks_stat

# Conditional Boundary FM

In [ ]:
width_range=(0.1, 5.0)

In [ ]:
class ConditionalBoundaryFlowMatcher(nn.Module):
    def __init__(self):
        super().__init__()
        # Input dim is 4: [x_t, t, dist_left, dist_right]
        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.SiLU(),
            nn.Linear(64, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 64),
            nn.SiLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_t, t, a, b):
        d_left = x_t - a
        d_right = b - x_t

        x_t = x_t.view(-1, 1)
        t = t.view(-1, 1)
        d_left = d_left.view(-1, 1)
        d_right = d_right.view(-1, 1)

        inputs = torch.cat([x_t, t, d_left, d_right], dim=-1)
        return self.net(inputs)

In [ ]:
def train_conditional_fm(dataloader, epochs=25, width_range=(0.1, 5.0)):
    model = ConditionalBoundaryFlowMatcher().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    loss_list = []

    print("Starting Conditional Flow Matching Training...")

    for epoch in tqdm(range(epochs)):
        model.train()
        total_loss = 0.0

        for batch in dataloader:
            x_1 = batch[0].to(device).view(-1, 1)  # real data
            batch_size = x_1.shape[0]

            x_0 = torch.randn_like(x_1)  # noise
            t = torch.rand((batch_size, 1), device=device)

            # Boundaries' width
            d = torch.empty((batch_size, 1), device=device).uniform_(*width_range)

            # Offset from min' boundary
            o = torch.rand((batch_size, 1), device=device) * d

            a = x_1 - o
            b = a + d

            x_t = (1 - t) * x_0 + t * x_1
            target_v = x_1 - x_0

            optimizer.zero_grad()
            pred_v = model(x_t, t, a, b)

            loss = criterion(pred_v, target_v)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        loss_list.append(avg_loss)

        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch}/{epochs} | Loss: {avg_loss:.6f}")

    print("Training Complete!")
    return model, loss_list

In [ ]:
batch_size = 2048
real_data_2d = real_data.view(-1, 1)
train_dataset = TensorDataset(real_data_2d)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

print(f"Total batches per epoch: {len(train_loader)}")
for batch in train_loader:
    x_1_batch = batch[0]
    print(f"Batch shape: {x_1_batch.shape}")
    break

In [ ]:
conditional_fm_model, loss_list_fm = train_conditional_fm(train_loader, epochs=100, width_range=width_range)

In [ ]:
plt.plot(loss_list_fm)
plt.show()

In [ ]:
def sample_trajectory(model, boundaries, n_samples, steps=100):
    x = torch.randn(n_samples, 1, device=device)
    dt = 1.0 / steps
    a, b = boundaries

    traj = [x.cpu()]
    model.eval()
    with torch.no_grad():
        for i in range(steps):
            t = i * dt
            t_batch = torch.full((n_samples, 1), t, device=device)
            v = model(x, t_batch, a, b)
            x = x + v * dt
            traj.append(x.cpu())

    return torch.stack(traj).numpy()  # Final shape: [steps + 1, n_samples, 2]

In [ ]:
boundaries1 = (-2, 2)
sampled_data1 = sample_trajectory(conditional_fm_model, boundaries=boundaries1, n_samples=100000)
hardflow_samples_on_base_fm1 = np.load(f"hardflow_samples_on_base_fm_{boundaries1[0]}_{boundaries1[1]}.npy")
hardflow_samples_on_base_fm1 = np.expand_dims(hardflow_samples_on_base_fm1, axis=0)

In [ ]:
visualize_fm(
    sampled_data=sampled_data1,
    primary_label="Constrained FM",
    secondary_sampled_data=hardflow_samples_on_base_fm1,
    secondary_label="Unconstrained FM + HF",
    true_data=real_data[:100000],
    title="Constrained FM vs Real Data",
    boundaries=boundaries1
)

In [ ]:
boundaries2 = (-4, 0)
sampled_data2 = sample_trajectory(conditional_fm_model, boundaries=boundaries2, n_samples=100000)
hardflow_samples_on_base_fm2 = np.load(f"hardflow_samples_on_base_fm_{boundaries2[0]}_{boundaries2[1]}.npy")
hardflow_samples_on_base_fm2 = np.expand_dims(hardflow_samples_on_base_fm2, axis=0)

In [ ]:
visualize_fm(
    sampled_data=sampled_data2,
    primary_label="Constrained FM",
    secondary_sampled_data=hardflow_samples_on_base_fm2,
    secondary_label="Unconstrained FM + HF",
    true_data=real_data[:100000],
    title="Constrained FM vs Real Data",
    boundaries=boundaries2
)

In [ ]:
boundaries3 = (0, 2.5)
sampled_data3 = sample_trajectory(conditional_fm_model, boundaries=boundaries3, n_samples=100000)
hardflow_samples_on_base_fm3 = np.load(f"hardflow_samples_on_base_fm_{boundaries3[0]}_{boundaries3[1]}.npy")
hardflow_samples_on_base_fm3 = np.expand_dims(hardflow_samples_on_base_fm3, axis=0)

In [ ]:
visualize_fm(
    sampled_data=sampled_data3,
    primary_label="Constrained FM",
    secondary_sampled_data=hardflow_samples_on_base_fm3,
    secondary_label="Unconstrained FM + HF",
    true_data=real_data[:100000],
    title="Constrained FM vs Real Data",
    boundaries=boundaries3
)

In [ ]:
boundaries4 = (-1, 1)
sampled_data4 = sample_trajectory(conditional_fm_model, boundaries=boundaries4, n_samples=100000)
hardflow_samples_on_base_fm4 = np.load(f"hardflow_samples_on_base_fm_{boundaries4[0]}_{boundaries4[1]}.npy")
hardflow_samples_on_base_fm4 = np.expand_dims(hardflow_samples_on_base_fm4, axis=0)

In [ ]:
visualize_fm(
    sampled_data=sampled_data4,
    primary_label="Constrained FM",
    secondary_sampled_data=hardflow_samples_on_base_fm4,
    secondary_label="Unconstrained FM + HF",
    true_data=real_data[:100000],
    title="Constrained FM vs Real Data",
    boundaries=boundaries4
)

# Adding Hardflow to fix the edges

In [ ]:
def sample_trajectory_hardflow(model, boundaries, n_samples, loss_fn, steps=100, guidance_scale=5.0, start_guidance_t=0.5):
    """
    HardFlow-style guided sampling for 1D Flow Matching, with delayed guidance.
    """
    # Assuming device is defined globally, or pass it as an arg
    device = next(model.parameters()).device
    x = torch.randn(n_samples, 1, device=device)
    dt = 1.0 / steps
    a, b = boundaries

    traj = [x.cpu()]
    model.eval()

    for i in range(steps):
        t = i * dt
        t_batch = torch.full((n_samples, 1), t, device=device)

        # --- GUIDED PHASE ---
        if t >= start_guidance_t:
            x = x.detach().requires_grad_(True)
            v = model(x, t_batch, a, b)

            # predict x_1
            x1_hat = x + (1 - t) * v

            # calculate the constraint loss on the predicted destination
            loss = loss_fn(x1_hat).sum()

            # calculate the gradient of the loss w.r.t the current position x_t
            grad = torch.autograd.grad(loss, x)[0]

            # apply guidance
            current_scale = t * guidance_scale
            v_guided = v - current_scale * grad

        # --- UNGUIDED PHASE (Fast) ---
        else:
            x = x.detach()  # No gradients needed yet
            with torch.no_grad():
                v = model(x, t_batch, a, b)

            # No gradient penalty applied
            v_guided = v

        # Euler step
        x = x.detach() + v_guided.detach() * dt

        traj.append(x.cpu())

    return torch.stack(traj).numpy()  # Final shape: [steps + 1, n_samples, 1]

## loss function 1

In [ ]:
def create_boundary_loss_relu(min_val, max_val):
    """
    Creates a loss function that penalizes points outside [min_val, max_val].
    Uses a ReLU to provide smooth, distance-proportional gradients.
    """
    def boundary_loss(x1_hat):
        # Penalty if x1_hat is LESS than min_val
        loss_lower = torch.nn.functional.relu(min_val - x1_hat)

        # Penalty if x1_hat is GREATER than max_val
        loss_upper = torch.nn.functional.relu(x1_hat - max_val)

        return loss_lower + loss_upper

    return boundary_loss

## loss function 2

In [ ]:
def create_boundary_loss_relu_squared(min_val, max_val):
    """
    Creates a loss function that penalizes points outside [min_val, max_val].
    Uses a squared ReLU to provide smooth, distance-proportional gradients.
    """
    def boundary_loss(x1_hat):
        # Penalty if x1_hat is LESS than min_val
        loss_lower = torch.nn.functional.relu(min_val - x1_hat) ** 2

        # Penalty if x1_hat is GREATER than max_val
        loss_upper = torch.nn.functional.relu(x1_hat - max_val) ** 2

        return loss_lower + loss_upper

    return boundary_loss

## loss function 3

In [ ]:
def create_boundary_loss_exponential_squared(min_val, max_val, steepness=2.0):
    """
    Creates a smooth exponential barrier.
    steepness controls how sharply the barrier rises.
    """
    def boundary_loss(x1_hat):
        # Repulsion from the lower boundary
        loss_lower = torch.exp(steepness * (min_val - x1_hat))**2

        # Repulsion from the upper boundary
        loss_upper = torch.exp(steepness * (x1_hat - max_val))**2

        return loss_lower + loss_upper

    return boundary_loss

## loss function 4

In [ ]:
def create_boundary_loss_softplus_squared(min_val, max_val, steepness=2.0):
    def boundary_loss(x1_hat):
        # softplus(x) = log(1 + exp(x))
        # It grows smoothly then becomes linear.
        loss_lower = torch.nn.functional.softplus(steepness * (min_val - x1_hat))**2
        loss_upper = torch.nn.functional.softplus(steepness * (x1_hat - max_val))**2

        return loss_lower + loss_upper
    return boundary_loss

## Sample

In [ ]:
sampled_data_hf1 = sample_trajectory_hardflow(
    model=conditional_fm_model,
    boundaries=boundaries1,
    n_samples=100000,
    loss_fn=create_boundary_loss_softplus_squared(*boundaries1),
    guidance_scale=0.5,
    start_guidance_t=0
)

In [ ]:
visualize_fm(
    sampled_data=sampled_data_hf1,
    secondary_sampled_data=sampled_data1,
    true_data=real_data[:100000],
    title="Constrained FM + HF vs Real GMM",
    boundaries=boundaries1
)

In [ ]:
sampled_data_hf2 = sample_trajectory_hardflow(
    model=conditional_fm_model,
    boundaries=boundaries2,
    n_samples=100000,
    loss_fn=create_boundary_loss_softplus_squared(*boundaries2),
    guidance_scale=0.5,
    start_guidance_t=0
)

In [ ]:
visualize_fm(
    sampled_data=sampled_data_hf2,
    secondary_sampled_data=sampled_data2,
    true_data=real_data[:100000],
    title="Constrained FM + HF vs Real GMM",
    boundaries=boundaries2
)

In [ ]:
sampled_data_hf3 = sample_trajectory_hardflow(
    model=conditional_fm_model,
    boundaries=boundaries3,
    n_samples=100000,
    loss_fn=create_boundary_loss_softplus_squared(*boundaries3),
    guidance_scale=0.5,
    start_guidance_t=0
)

In [ ]:
visualize_fm(
    sampled_data=sampled_data_hf3,
    secondary_sampled_data=sampled_data3,
    true_data=real_data[:100000],
    title="Constrained FM + HF vs Real GMM",
    boundaries=boundaries3
)

In [ ]:
sampled_data_hf4 = sample_trajectory_hardflow(
    model=conditional_fm_model,
    boundaries=boundaries4,
    n_samples=100000,
    loss_fn=create_boundary_loss_softplus_squared(*boundaries4),
    guidance_scale=0.5,
    start_guidance_t=0
)

In [ ]:
visualize_fm(
    sampled_data=sampled_data_hf4,
    secondary_sampled_data=sampled_data4,
    true_data=real_data[:100000],
    title="Constrained FM + HF vs Real GMM",
    boundaries=boundaries4
)